In [29]:
from langgraph.graph import StateGraph, START , END
from langgraph.graph.message import add_messages
from typing import Annotated,TypedDict
from langchain_core.messages import HumanMessage, BaseMessage
import asyncio
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from dotenv import load_dotenv
import os

load_dotenv()

llm=HuggingFaceEndpoint(
    # repo_id="HuggingFaceH4/zephyr-7b-beta",
    # repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    # repo_id="zai-org/GLM-4.7-Flash",
    repo_id="openai/gpt-oss-20b",
    # repo_id="openai/gpt-oss-120b",
    # repo_id="zlyngkhoi/qwen2_lrp_lora_3b",
    # repo_id="lmsys/vicuna-13b-v1.5",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm)
# generator=ChatHuggingFace(llm=llm)

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient

client=MultiServerMCPClient(
    {
        "math":{
            "transport":"stdio",
            "command":"python3",
            "args":["/home/abrar/Desktop/Abrar/LangGraph/09_MCP/main.py"]
        },
        # "expense_tracker":{
        #     "transport":"stdio",
        #     "command":"python3",
        #     "args":["/home/abrar/Desktop/Abrar/MCP/ex_tracker/main.py"]
        # }
        # "expense_tracker":{
        #     "transport":"sse",
        #     "url":"http://localhost:6277/sse",
        #     "headers":{
        #         "MCP_PROXY_AUTH_TOKEN":"Bearer 9253e95f0c22ee66c3aac8434dd1350e0193408a96742eeffbc8a4dc4cb838dc"
        #     }
        # }
        "expense_tracker":{
            "transport":"streamable_http",
            "url":"http://127.0.0.1:8000/mcp"
        },
        "github": {
            "transport": "streamable_http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {os.getenv('GITHUB_PAT')}"
            }
        }
    }
)
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


async def build_graph():

    tools=await client.get_tools()

    github_tools=tools[-1]
    for tool in github_tools:
        print(tool[1])
        # break

    bound_model=model.bind_tools(tools)

    async def chat_node(state:ChatState):
        messages=state['messages']
        response=await bound_model.ainvoke(messages)

        return {
            'messages':[response]
        }
    
    # tools=[tools]
    
    tool_node=ToolNode(tools)

    graph=StateGraph(ChatState)

    graph.add_node('chat_node',chat_node)
    graph.add_node('tools',tool_node)

    graph.add_edge(START,'chat_node')
    graph.add_conditional_edges('chat_node',tools_condition)
    graph.add_edge('tools',"chat_node")

    chatbot=graph.compile()

    return chatbot


async def main():
    chatbot=await build_graph()

    result=await chatbot.ainvoke({"messages":[HumanMessage(content="can you check my all expenses for current month")]})
    
    # ainvoke -=> Asynchronous invoke
    
    print(result['messages'][-1].content)

    


In [30]:
if __name__=="__main__":
    await main()

update_pull_request_branch
Update the branch of a pull request with the latest changes from the base branch.
{'type': 'object', 'properties': {'expectedHeadSha': {'type': 'string', 'description': "The expected SHA of the pull request's HEAD ref"}, 'owner': {'type': 'string', 'description': 'Repository owner'}, 'pullNumber': {'type': 'number', 'description': 'Pull request number'}, 'repo': {'type': 'string', 'description': 'Repository name'}}, 'required': ['owner', 'repo', 'pullNumber']}
False
False
None
None
{'title': 'Update pull request branch', 'readOnlyHint': None, 'destructiveHint': None, 'idempotentHint': None, 'openWorldHint': None}
False
False
content_and_artifact
None
None
<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7dbdf8ec5260>
Here’s a quick snapshot of all the expenses you’ve logged for this month (May 2026):

| Category | Total (₹) |
|----------|-----------|
| Food     | 400.00 |

**Overall total for May 2026:** ₹400.00  
- *Top category:* Food (1